In [85]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [86]:
import sdv
print(sdv.version.public)

1.20.1


In [87]:
data = pd.read_csv('/kaggle/input/titanic/train.csv')

In [88]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [89]:
data.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [90]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [91]:
print((data['Age'] <= 18).sum())

139


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning:

invalid value encountered in less_equal



In [92]:
#Age - fill with median according to age and pclass
data['Age'] = data.groupby(['Sex', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))

# Missing information for cabin 
data['Cabin'] = data['Cabin'].fillna("Missing")

# Embarked - fill with dominant 
data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])

An SDV synthesizer is an object that we can use to create synthetic data. It learns patterns from the real data and replicates them to generate synthetic data.

In [101]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        891 non-null    object 
 11  Embarked     891 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [93]:
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()

# Tworzenie metadanych
metadata = SingleTableMetadata()

# Automatyczne wykrycie typów kolumn
metadata.detect_from_dataframe(data=data)

metadata.validate()

In [94]:
from sdv.single_table import GaussianCopulaSynthesizer

synthesizer = GaussianCopulaSynthesizer(metadata)

synthesizer.fit(data)

synthetic_data = synthesizer.sample(891)

print(synthetic_data.head())

/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:144: FutureWarning:

The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.

/usr/local/lib/python3.11/dist-packages/sdv/single_table/base.py:122: UserWarning:

We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.



   PassengerId  Survived  Pclass                                        Name  \
0      4326007         1       2                           Beane, Mr. Edward   
1      4069891         0       3  Sedgwick, Mr. Charles Frederick Waddington   
2      2697777         0       3                         Toomey, Miss. Ellen   
3     10119885         0       3                         Ilett, Miss. Bertha   
4      4368626         0       3                        Youseff, Mr. Gerious   

      Sex    Age  SibSp  Parch         Ticket     Fare    Cabin Embarked  
0  female  46.51      0      0         113760  35.2644  Missing        S  
1    male  19.73      0      0          19877   5.3434  Missing        S  
2    male  61.40      0      0         250655   7.0772  Missing        S  
3  female   4.79      1      0         370372  43.1792  Missing        S  
4    male  32.36      0      0  SC/PARIS 2167  21.5023  Missing        S  


SDV's diagnostic performs some basic checks such as:

All primary keys must be unique
Continuous values must adhere to the min/max of the real data
Discrete columns (non-PII) must have the same categories as the real data
Etc.

In [95]:
from sdv.evaluation.single_table import run_diagnostic

diagnostic = run_diagnostic(
    real_data = data,
    synthetic_data = synthetic_data,
    metadata = metadata
)

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 12/12 [00:00<00:00, 1147.53it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 236.35it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



In [96]:
from sdv.evaluation.single_table import evaluate_quality

quality_report = evaluate_quality(
    data,
    synthetic_data,
    metadata
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 300.42it/s]|
Column Shapes Score: 89.52%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 201.25it/s]|
Column Pair Trends Score: 66.44%

Overall Score (Average): 77.98%



In [97]:
quality_report.get_details('Column Shapes'), #name and ticket is the lowest 

(      Column        Metric     Score
 0   Survived  TVComplement  0.994388
 1     Pclass  TVComplement  0.975309
 2       Name  TVComplement  0.626263
 3        Sex  TVComplement  0.998878
 4        Age  KSComplement  0.852974
 5      SibSp  TVComplement  0.968575
 6      Parch  TVComplement  0.989899
 7     Ticket  TVComplement  0.677890
 8       Fare  KSComplement  0.854097
 9      Cabin  TVComplement  0.922559
 10  Embarked  TVComplement  0.986532,)

In [98]:
from sdv.evaluation.single_table import get_column_plot

fig = get_column_plot(
    real_data = data,
    synthetic_data = synthetic_data,
    column_name = "Age",
    metadata = metadata
)
fig.show()

#Median method had a strong influence on a data

In [100]:
fig = get_column_plot(
    real_data = data,
    synthetic_data = synthetic_data,
    column_name = "Parch",
    metadata = metadata
)
fig.show()